# Feature Scaling: Standardization

Standardization is a feature scaling technique that rescales a column so its mean becomes 0 and its standard deviation becomes 1. It is also called Z-score normalization.

$$z = \frac{x - \mu}{\sigma}$$

We already used this exact formula in the normal distribution notebook. Here we use it as a preprocessing step before training a model.

## Why Do We Need Feature Scaling?

Many ML algorithms measure the distance between two data points. KNN is the classic example. The distance formula is:

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

Now take two people, where x is salary and y is age:

* Person 1: salary 48000, age 27
* Person 2: salary 83000, age 50

The salary part of the distance is:

$$(83000 - 48000)^2 = 1{,}225{,}000{,}000$$

The age part is:

$$(50 - 27)^2 = 529$$

Salary is about 2 million times bigger than age in this sum. The distance is basically just salary, and age is ignored. That is not because salary matters more. It is only because salary has bigger numbers.

Feature scaling fixes this by putting all columns on a similar scale, so every feature gets a fair say.

## Does Standardization Really Help? A Test

To see the effect, I will train the same classifier twice, once on raw data and once on standardized data, and compare the accuracy.

## A Small Dataset to See Standardization Work

I made a dataset of 20 people with their age, salary, and whether they are married (1) or not (0). Age is in tens and salary is in tens of thousands, so salary will dominate any distance based model like KNN.

In [15]:
import pandas as pd

people = pd.DataFrame({
    'age':    [22, 24, 25, 26, 27, 28, 29, 30, 31, 32,
               33, 34, 35, 36, 38, 40, 41, 43, 45, 23],
    'salary': [28000, 35000, 42000, 38000, 60000, 55000, 72000, 48000, 90000, 65000,
               120000, 45000, 85000, 70000, 150000, 52000, 30000, 110000, 95000, 140000],
    'married': [0, 0, 0, 0, 0, 1, 1, 0, 1, 1,
                1, 0, 1, 1, 1, 1, 0, 1, 1, 0]
})

people.head()

,age,salary,married
0,22,28000,0
1,24,35000,0
2,25,42000,0
3,26,38000,0
4,27,60000,0


In [16]:
people.describe().T[['mean', 'std', 'min', 'max']]

,mean,std,min,max
age,32.10,6.827961,22.0,45.0
salary,71500.00,36120.412657,28000.0,150000.0
married,0.55,0.510418,0.0,1.0


## Train Test Split

Now I will split the 20 people into a training set and a test set. The model learns from the training set and gets scored on the test set, which it has never seen.

I am using `stratify` so both sets keep the same mix of married and not married people. With only 20 rows, this matters.

In [49]:
from sklearn.model_selection import train_test_split

X = people[['age', 'salary']]
y = people['married']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


## Standardizing the Data

Now I will standardize the age and salary columns with scikit-learn's `StandardScaler`. It applies the same z-score formula we used before, $z = \frac{x - \mu}{\sigma}$, to every column.

In [50]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_train = scaler.fit_transform(X_train)

scaled_test = scaler.transform(X_test)

scaled_train_df = pd.DataFrame(scaled_train, columns=['age', 'salary'])
scaled_test_df = pd.DataFrame(scaled_test, columns=['age', 'salary'])

scaled_test_df

,age,salary
0,-0.797997,-0.942318
1,0.302689,-0.730448
2,-0.110069,0.631575
3,0.577860,0.026232
4,0.853032,2.447605


In [56]:
print(scaled_train_df)
print('-'*30)
print(scaled_test_df)

         age    salary
0  -0.522826 -0.427776
1   1.816132  0.782911
2  -0.385240  0.086766
3  -1.348340 -1.244989
4   0.165103  1.539590
5  -0.660412 -0.276440
6  -0.247654 -0.639646
7   0.027517 -0.125104
8   0.440274  0.480239
9  -0.935583 -0.821249
10  1.540960  1.236918
11  1.265789 -1.184455
12 -1.210754  2.144933
13  1.128203 -0.518577
14 -1.073169 -1.033119
------------------------------
        age    salary
0 -0.797997 -0.942318
1  0.302689 -0.730448
2 -0.110069  0.631575
3  0.577860  0.026232
4  0.853032  2.447605


In [60]:
# Verifying: Standadization has mean of 0 and std 1.

round(scaled_train_df.describe().T[['mean', 'std']], 1)

,mean,std
age,-0.0,1.0
salary,0.0,1.0


## Checking the Scaled Training Data

After scaling, both columns have a mean of 0 and a std of about 1. Age and salary are now on the same scale, so neither one can dominate a distance calculation.

## Why Is Standardization Important?

Let's check why standardization matters with a real dataset. The wine dataset has 13 numeric columns. One of them, `proline`, has values in the hundreds and thousands, while most of the others are below 20.

KNN works on distance, so `proline` will dominate it. I will train the same KNN model on raw data and on standardized data, then compare the accuracy.

In [64]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

wine = load_wine(as_frame=True)
X = wine.data
y = wine.target

X

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0


In [65]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# raw
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
print("Raw accuracy:", accuracy_score(y_test, knn_raw.predict(X_test)))

# scaled (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
print("Scaled accuracy:", accuracy_score(y_test, knn_scaled.predict(X_test_scaled)))

Raw accuracy: 0.7777777777777778
Scaled accuracy: 0.9333333333333333


## Result

* Raw accuracy: **0.77**
* Scaled accuracy: **0.93**

The test set has 45 rows, so this means 35 correct predictions on raw data and 42 on scaled data. Same model, same split. The only thing I changed was the scaling.

On raw data, `proline` is so big that KNN mostly ignores the other 12 columns. After standardizing, every column has the same spread, so all of them get a fair say in the distance.

I fit the scaler on the training data only and used `transform` on the test data, so the test rows stayed unseen.

So standardization is important for distance based models like KNN. This is the same problem as the salary vs age example, just on real data.